In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
import joblib

# Load the dataset
file_path = "mpox.csv"  # Ensure this file is in the working directory
df = pd.read_csv(file_path)

# Drop the ID column as it is not needed for training
df_cleaned = df.drop(columns=["ID"])

# Separate features and target variable
X = df_cleaned.drop(columns=["Status"])
y = df_cleaned["Status"]

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Feature Selection using RFE
logreg = LogisticRegression(max_iter=200, class_weight='balanced')
rfe = RFE(logreg, n_features_to_select=20)  # Select top 20 features
X_train_rfe = rfe.fit_transform(X_train_resampled, y_train_resampled)
X_test_rfe = rfe.transform(X_test)  # Transform test data with selected features

# Train Logistic Regression model on selected features
model = LogisticRegression(max_iter=200, class_weight='balanced')
model.fit(X_train_rfe, y_train_resampled)

# Export the trained model
joblib.dump(model, 'logistic_regression_model.pkl')

# Make predictions
y_pred = model.predict(X_test_rfe)

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=1)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, zero_division=1)

# Print key performance metrics
print("Model Performance:")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Show selected features
selected_features = X.columns[rfe.support_]
print("Selected Features:")
print(selected_features)


Model Performance:
Accuracy: 0.81
Precision: 0.97
Recall: 0.82
F1-score: 0.89
Confusion Matrix:
[[ 4  1]
 [ 7 31]]
Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.80      0.50         5
           1       0.97      0.82      0.89        38

    accuracy                           0.81        43
   macro avg       0.67      0.81      0.69        43
weighted avg       0.90      0.81      0.84        43

Selected Features:
Index(['rash', 'skin lesions', 'ulcerative lesions', 'oral and genital ulcers',
       'fever', 'genital ulcer lesions', 'blisters', 'fatigue', 'dysphagia',
       'decreased physical strength', 'chills', 'adenomegaly', 'myalgia',
       'swollen lymph nodes', 'sore throat', 'malaise', 'loss of appetite',
       'Vesicles', 'encephalitis', 'blisters on limbs and genitals'],
      dtype='object')


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
import joblib
from flask import Flask, request, jsonify

# Load the dataset
file_path = "mpox.csv"  # Ensure this file is in the working directory
df = pd.read_csv(file_path)

# Drop the ID column as it is not needed for training
df_cleaned = df.drop(columns=["ID"])

# Separate features and target variable
X = df_cleaned.drop(columns=["Status"])
y = df_cleaned["Status"]

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Feature Selection using RFE
logreg = LogisticRegression(max_iter=200, class_weight='balanced')
rfe = RFE(logreg, n_features_to_select=20)  # Select top 20 features
X_train_rfe = rfe.fit_transform(X_train_resampled, y_train_resampled)
X_test_rfe = rfe.transform(X_test)  # Transform test data with selected features

# Train Logistic Regression model on selected features
model = LogisticRegression(max_iter=200, class_weight='balanced')
model.fit(X_train_rfe, y_train_resampled)

# Export the trained model
joblib.dump(model, 'monkeypox_model.pkl')

# Make predictions
y_pred = model.predict(X_test_rfe)

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=1)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, zero_division=1)

# Print key performance metrics
print("Model Performance:")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)

# Show selected features
selected_features = X.columns[rfe.support_]
print("Selected Features:")
print(selected_features)

# Create Flask app
app = Flask(__name__)

# Load the model
model = joblib.load('monkeypox_model.pkl')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        features = pd.DataFrame([data['features']])
        prediction = model.predict(features)
        return jsonify({'prediction': int(prediction[0])})
    except Exception as e:
        return jsonify({'error': str(e)})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)


Model Performance:
Accuracy: 0.81
Precision: 0.97
Recall: 0.82
F1-score: 0.89
Confusion Matrix:
[[ 4  1]
 [ 7 31]]
Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.80      0.50         5
           1       0.97      0.82      0.89        38

    accuracy                           0.81        43
   macro avg       0.67      0.81      0.69        43
weighted avg       0.90      0.81      0.84        43

Selected Features:
Index(['rash', 'skin lesions', 'ulcerative lesions', 'oral and genital ulcers',
       'fever', 'genital ulcer lesions', 'blisters', 'fatigue', 'dysphagia',
       'decreased physical strength', 'chills', 'adenomegaly', 'myalgia',
       'swollen lymph nodes', 'sore throat', 'malaise', 'loss of appetite',
       'Vesicles', 'encephalitis', 'blisters on limbs and genitals'],
      dtype='object')
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://26.26.26.1:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Apr/2025 23:30:00] "POST /predict HTTP/1.1" 200 -
